# Libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd

# Data Import

In [3]:
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_gas_sorted_by_sql.csv')
nutrition = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_06_09_trial_database_curation/subsets_1_2_3_4_nutrition_sorted_by_sql.csv')

In [4]:
gas.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
0,1,3,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,24.237242,227.8476704,37.051645,65.414752,NaN,"1,2,3",NaN,NaN,NaN,NaN
1,1,4,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,24.362056,270.0095702,38.185077,63.799940,NaN,"1,2,3",NaN,NaN,NaN,NaN


In [8]:
gas.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_8h_percentage', 'ch4_24h_percentage',
       'gas_ml_g_dm_incubated_24h', 'part_fact', 'ch4_ml_g_dm_incubaed_24h',
       'ch4_ml_g_ndf_digested_24h', 'methane_intensity', 'tddm',
       'information_remarks_1', 'information_remarks_2', 'gas_remarks_1',
       'gas_remarks_2', 'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [5]:
nutrition.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


# Curation

In [6]:
remarks_columns = ['information_remarks_1', 'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2']

# Convert columns to string type and check for 'elim' or 'descar' (case-insensitive)
conditions = []
for col in remarks_columns:
    contains_elim = gas[col].astype(str).str.contains('elim', na=False, case=False)
    contains_descar = gas[col].astype(str).str.contains('descar', na=False, case=False)
    conditions.append(contains_elim | contains_descar)

# Combine the remarks conditions using OR logic (any() along axis=1)
# Create a boolean series where True means 'elim' or 'descar' is found in at least one remark column
has_keywords = pd.concat(conditions, axis=1).any(axis=1)

# New condition: check if 'methane_intensity' is NaN
is_methane_intensity_null = gas['methane_intensity'].isna()

# Combine all conditions for deletion using OR logic
final_delete_condition = has_keywords | is_methane_intensity_null

# Create the 'delete' column, assigning 'yes' or 'no' based on the final condition
gas['delete'] = np.where(final_delete_condition, 'yes', 'no')

In [11]:
import pandas as pd

# Count the number of 'valid' rows (where 'delete' is 'no') for each id_lab and batch combination
valid_rows_per_id_lab_batch = gas[gas['delete'] == 'no'].groupby(['id_lab', 'batch']).size()

# Identify (id_lab, batch) combinations that have less than 2 valid rows.
# These combinations should be entirely excluded.
id_lab_batch_to_completely_exclude = valid_rows_per_id_lab_batch[valid_rows_per_id_lab_batch < 2].index

# Create a multi-index from the gas DataFrame to easily filter
gas_multi_index = gas.set_index(['id_lab', 'batch'])

# Step 1: Exclude all rows from the identified (id_lab, batch) combinations
gas_filtered_by_batch_count = gas_multi_index.drop(id_lab_batch_to_completely_exclude, errors='ignore').reset_index()

# Step 2: From the remaining data, exclude individual rows where 'delete' is 'yes'
gas_clean = gas_filtered_by_batch_count[gas_filtered_by_batch_count['delete'] == 'no']

In [ ]:
gas.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_complete_subsets_1234_2026_06_09.csv', index=None)

In [12]:
gas_clean.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/gas_clean_subsets_1234_2026_06_09.csv', index=None)

In [ ]:
nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_06_09_trial_database_curation/nutrition_complete_1234_2026_06_09.csv', index=None)